# Data Preprocessing

The tensors of the previous section were made by our own code, filled with
zeros, ones, or random numbers in the shape we chose. Data from the world
rarely arrive that way. A table of records is usually stored as a text file,
with gaps where a value was not recorded and with words where a model needs
numbers. Turning such a table into tensors is this section's subject, and it
takes more judgment than the word *preprocessing* suggests.

Our example is a table of ten home sales, each recorded with the number of
rooms, the type of roof, the floor area, and the sale price. Small as it is,
it has everything that makes real tables awkward. Three of the ten homes lack
the number of rooms, and four lack the roof type. The roof type is a word,
such as `Slate` or `Tile`, not a number. And the floor area, measured in
square feet, is hundreds of times larger than the number of rooms. Before a
model sees this table we have to decide what to put in the gaps, how to turn
words into numbers, and whether to bring the columns to a common scale.

These decisions change what a model can learn. Dropping every incomplete row
would throw away more than half of the table. Numbering slate roofs, tile
roofs, and missing entries 0, 1, and 2 would invent an order among them that
does not exist. And left unscaled, the floor area dominates any computation
that compares homes, so that the number of rooms barely counts. This section
turns the table into a tensor of inputs and a tensor of targets with the
[pandas](https://pandas.pydata.org/) library
[@McKinney.2010], the standard Python tool for loading, inspecting, and
transforming tables (the figure).

![From a file to tensors. `read_csv` loads the text file into a DataFrame with gaps and text columns; imputation, encoding, and scaling make every entry a number; `to_numpy` hands the result to the framework as the tensors `X` and `y`.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/pandas-pipeline.svg)

We import pandas under its usual alias `pd`, the module `os` for file paths,
and the deep learning framework, which we need only at the end, for the
conversion to tensors.

In [ ]:
import os
import pandas as pd
import tensorflow as tf

## Loading and Inspecting

### CSV Files

A *comma-separated values* (CSV) file stores a table as plain text. Each line
holds one row, the first line usually holds the column names, and commas
separate the fields within a line. We write the ten home sales to the file
`../data/house_tiny.csv`, with the columns `NumRooms`, `RoofType`, `Area`
(in square feet), and `Price` (in dollars). An empty field marks a missing
entry.

In [ ]:
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('''NumRooms,RoofType,Area,Price
3,Slate,1500,210000
,Tile,2100,290000
2,,850,127500
4,Slate,1940,258000
,,1200,168000
3,Tile,1650,225000
5,Slate,2600,375000
2,,900,142000
,Tile,1750,240000
4,,2050,295000''')

The function `read_csv` parses the file into a *DataFrame*, the table object
of pandas. A DataFrame consists of named columns, each holding values of one
type, and it labels its rows; the numbers 0 to 9 at the left of the output
are these labels.

In [ ]:
data = pd.read_csv(data_file)
data

Every empty field has become `NaN` (not a number), the marker that pandas
uses for a *missing value*.

### Data Types and Summaries

Before transforming a table, it pays to check what pandas made of it: how
many rows and columns it has, how many entries of each column are present,
and which data type pandas inferred for each column. The method `info`
reports all three.

In [ ]:
data.info()

The table has 10 rows and 4 columns. `NumRooms` has 7 entries present and
`RoofType` 6, so 3 and 4 entries are missing. The data types show how pandas
read each column. `Area` and `Price` hold 64-bit integers. `NumRooms` holds
whole numbers too, and yet pandas has made it a `float64` column. The gaps
are to blame: `NaN` is a floating-point value, and an integer column cannot
hold it. `RoofType`
has the type `str`, which pandas uses for text (versions before pandas 3.0
used `object`). A text column usually holds categories, as here, and must be
turned into numbers before a model can use it.

For the numerical columns, the method `describe` summarizes the values: how
many are present, their mean and standard deviation, and their quantiles.

In [ ]:
data.describe()

The columns have very different scales. The number of rooms ranges from 2 to
5, the area from 850 to 2,600 square feet, and the price from 127,500 to
375,000 dollars. The floor area averages 1,654 square feet, about 500 times
the average number of rooms, 3.3. We return to this difference when we
rescale the columns.

### Inputs and Targets

We want to predict the price of a home from its other attributes. The price is
the *target*, also called the *label*, and the remaining columns are the
inputs, also called *features*. We separate the two first, so that the steps
below transform only the inputs and the price never ends up among them. The
indexer `iloc` selects by position, with the slicing rules of tensors:
`data.iloc[:, :3]` takes all rows and the first three columns, and
`data.iloc[:, 3]` takes the fourth column. The names of the selected
columns confirm the split:

In [ ]:
inputs, targets = data.iloc[:, :3], data.iloc[:, 3]
list(inputs.columns), targets.name

Selection by position is short, but it breaks without an error when a new
version of the file inserts or reorders columns. Selection by name, as in
`data.drop(columns='Price')`, does not; an exercise at the end of the
section takes this up.

## Cleaning and Encoding

### Missing Values

Values go missing in real data for many reasons: a field was not recorded,
did not apply, or was lost. Two methods count the gaps. `isna` returns a
table of the same shape that holds `True` where an entry is missing and
`False` elsewhere, and `sum` adds up each column, counting `True` as 1.

In [ ]:
inputs.isna().sum()

There are three common responses, and each gives something up.

* *Deletion* drops every row, or every column, that contains a missing
  value. It is simple, but it discards data. If values are missing more
  often for some kind of home, such as small ones, deletion also biases
  what remains.
* *Imputation* fills each gap with an estimate, such as the mean of the
  column. It keeps every row, but the filled values are guesses, and they
  make the column look less spread out than it is.
* An *indicator* is an extra column that records which values were missing,
  usually added alongside imputation. With it, a model can learn whether a
  missing value carries information of its own, for example when the number
  of rooms is missing mostly for large homes. The cost is one more column
  per feature, which a small table can ill afford.

Let us see what deletion would do to our table. Dropping every row that has
a missing entry keeps only the complete rows:

In [ ]:
len(inputs.dropna()), len(inputs)

Only 4 of the 10 homes remain, far too few, so we impute instead. For a
numerical column, the simplest estimate is the column mean; the median is
the safer choice when a few very large values pull the mean away from the
typical value. The
method `fillna` fills the gaps of each column with the value given for that
column, and `mean(numeric_only=True)` computes the mean of each numerical
column, skipping its missing entries. A text column has no mean, so the gaps
in `RoofType` remain; we deal with them when we encode the column.

In [ ]:
inputs = inputs.fillna(inputs.mean(numeric_only=True))
inputs

The three missing room counts are now 3.29, the mean $23/7$ of the seven
observed counts. No home has 3.29 rooms, of course. An imputed value is a
placeholder, and we meet these three again when we rescale the column.

### Categorical Features

The column `RoofType` holds *categories*: each home belongs to one of a few
groups, and the groups have no order. Counting a missing entry as a group of
its own, there are three groups: `Slate`, `Tile`, and missing. A model needs
numbers, and the numbering 0, 1, and 2 is the first thing that comes to
mind. Unfortunately, it does more than invent an order. It also places a
missing roof twice as far from slate as tile is, as though not knowing the
roof were an extreme case of tile.

*One-hot encoding* avoids both defects. It replaces the column by one new
column per category, which holds 1 for the homes in that category and 0 for
all others, so that any two categories are equally far apart. The function
`get_dummies` performs the encoding. With `dummy_na=True`, it treats a
missing value as one more category, which is the indicator strategy applied
to a categorical column. With `dtype=float`, the new columns hold the
numbers 0.0 and 1.0 rather than the values `False` and `True`.

In [ ]:
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=float)
inputs

`RoofType` has become the three columns `RoofType_Slate`, `RoofType_Tile`,
and `RoofType_nan`, and each row has a single 1 among them. One of the three
is therefore redundant, and for a column with only two values a single 0/1
column suffices. The numerical columns pass through unchanged, so every
column of `inputs` now holds numbers.

One-hot encoding suits columns with a few categories. A column with
thousands of distinct values, such as a postal code, would become thousands
of columns that are almost entirely zero. Such columns are better
represented by *embeddings*, short vectors of numbers that a model learns
for each value; we meet them in that section.

## Scaling and Converting

### Standardization

The two numerical columns still measure different things in different
units. In raw units, two homes with the same area that differ by two rooms
are closer to each other than two homes with the same number of rooms that
differ by three square feet. Any
computation that adds up differences between features, such as the distance
between two homes, is then decided almost entirely by the area. A feature on
a much larger scale than the others also slows down training, for a reason
that that section gives.

The standard remedy is *standardization*. We replace every entry $x$ of a
column by

$$\frac{x - \mu}{\sigma},$$

where $\mu$ is the mean of the column and $\sigma$ its standard deviation,
the typical distance of an entry from the mean. For a column with entries
$x_1, \ldots, x_n$, pandas computes it as $\sigma =
\sqrt{\frac{1}{n-1}\sum_{i=1}^n (x_i - \mu)^2}$. (Dividing by $n - 1$
instead of $n$ is a convention that that section explains.) After this
step every column has mean 0 and standard deviation 1, and an entry says how
many standard deviations it lies above or below the mean of its column. The
column no longer remembers whether it was measured in square feet or in
square meters. The one-hot columns already share a scale, 0 and 1, so we
leave them as they are.

In [ ]:
numerical = ['NumRooms', 'Area']
mu, sigma = inputs[numerical].mean(), inputs[numerical].std()
inputs[numerical] = (inputs[numerical] - mu) / sigma
inputs

The largest home, in row 6, has 5 rooms and 2,600 square feet. After
standardization these entries are 1.89 and 1.70: in both columns the home
lies 1.7 to 1.9 standard deviations above the mean, and the two numbers can
now be compared directly. The imputed room counts have become 0, because
mean imputation placed them at the mean of the column. The placeholders
have left a trace, however. Imputation shrank the standard deviation of
`NumRooms` from 1.11, that of the seven observed counts, to 0.91. The three
imputed entries sit at the mean and add
no spread, so every standardized room count moves farther from 0. With the
observed spread, the largest home would score 1.54 instead of 1.89.

One shortcut of this section should not be copied. All statistics, the means
used for imputation and the means and standard deviations used for
standardization, were computed from the whole
table, with no homes held out. In practice we hold back part of the data to
test a trained model, and the test has to imitate data that the model has never
seen. The statistics must therefore be computed from the training data alone
and then applied, unchanged, to the test data. Computing them from all the data
lets information about the test data *leak* into training, and the test then
overstates how well the model will do on new data. that section
follows this rule on a real dataset of home prices.

### Conversion to Tensors

What remains is a change of container. The method `to_numpy` converts a
DataFrame to a NumPy array, and the framework's tensor
constructor converts the array to a tensor. A DataFrame
stores floating-point columns as 64-bit numbers, whereas deep learning
frameworks compute in 32-bit floating point by default, so we ask `to_numpy`
for `float32` directly.

In [ ]:
X = tf.constant(inputs.to_numpy(dtype='float32'))
y = tf.constant(targets.to_numpy(dtype='float32'))
X.shape, X.dtype, y.shape, y.dtype

`X` has one row per home and one column per feature: the two standardized
numerical features and the three indicators of the roof type. `y` holds the
ten prices.

Ten homes are, admittedly, a toy. The pandas library holds a whole table in
main memory and processes it on the CPU, which suits the tables in this
book. Tables larger than memory need tools that read them in pieces, and
images, audio, and text need pipelines of their own, which later chapters
build. The tensor `X` is a matrix with one row per example, the form in
which models receive their data. What they do with such a matrix is the
subject of the next section.

## Summary

Real tables have missing entries, text columns, and columns on very
different scales, whereas a model needs a tensor of numbers. The pandas
library loads such a table into a DataFrame, and `info` and `describe` show its size, its
gaps, its data types, and the ranges of its columns. Missing values can be
deleted, imputed, or flagged with an indicator column; on small data,
imputation keeps the rows that deletion would discard. Categorical columns
become one 0/1 column per category, which avoids inventing an order, and
numerical columns are standardized to mean 0 and standard deviation 1, with
statistics computed from the training data only. `to_numpy` then hands the
table to the framework as a `float32` tensor.

## Exercises

1. [code] **Indexing by name.** This section splits `data` by position
   with `data.iloc[:, :3]` and `data.iloc[:, 3]`. Suppose a new version of
   `house_tiny.csv` adds a column `YearBuilt` between `Area` and `Price`.
    1. State what the positional split then puts into `inputs` and
       `targets`, and whether any later line of the section's code raises
       an error.
    1. Rewrite the split with column names so that `targets` holds `Price`
       and `inputs` holds every other column, for the new file and for the
       original one. The pandas user guide on
       [indexing](https://pandas.pydata.org/docs/user_guide/indexing.html)
       lists the options.

1. [code] **Imputation strategies.** Fill the missing values in `NumRooms`
   in three ways: with the column mean, with the column median, and with
   the column mean plus a "was-missing" indicator column.
    1. For the mean fill and the median fill, compute the mean and the
       standard deviation of the filled column, compare them with those of
       the observed values, and explain the direction of each change.
    1. Suppose that in a larger dataset `NumRooms` is missing mostly for
       large homes. State what a model sees for those homes after mean
       imputation alone. Then explain how a model that predicts the price
       as a weighted sum of its features plus a constant can use the
       indicator column to correct its predictions for them.
    1. State what each of the three treatments assumes about why the values
       are missing.

    *Adapted from Kaggle Learn,
    [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning),
    Exercise 2.*

1. **High-cardinality categories.** One-hot encoding turns a categorical
   column with $k$ distinct values into $k$ columns, plus one more with
   `dummy_na=True`.
    1. A table of $n = 10^6$ homes has a `ZipCode` column with no missing
       entries and $k = 10^4$ distinct values. Compute how many bytes its
       one-hot columns occupy when stored densely as `float64`, and how many
       of their entries are nonzero.
    1. A column `ListingID` holds a value unique to each row. Consider three
       treatments: one-hot encoding, an embedding (a learned vector for each
       distinct value), and dropping the column. For each, state what a
       model could learn about a home whose `ListingID` does not occur in
       the training data, and choose one treatment.

    *Adapted from Kaggle Learn,
    [Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning),
    Exercise 3.*

1. [code] **Where leakage hides.** This section fills the missing
   `NumRooms` entries with the mean of all observed values and
   standardizes `NumRooms` and `Area` with the mean and standard deviation
   of all ten rows. Treat the first eight rows of `data` as training data
   and the last two as test data.
    1. Redo the imputation and the standardization with statistics
       computed on the training rows only and applied unchanged to the test
       rows. Compare the standardized test rows with those of the
       whole-dataset recipe.
    1. Explain what the whole-dataset recipe lets the model learn about the
       test rows that the training-only recipe does not.
    1. Suppose a feature is constant on the training rows but not on the
       test rows. State what the training-only recipe produces for that
       feature on each split, and propose a fix.

1. [extended] **A real messy dataset, end to end.** Take a messy public
   dataset, for example the San Francisco restaurant-inspection data used
   in Berkeley's Data 100: one row per inspected business, with its name,
   address, zip code, latitude and longitude, and the scores of its health
   inspections, where an unknown latitude or longitude is recorded as
   $-9999$ rather than as a blank field. Run the full pipeline of this
   section: detect such a non-standard missing-value sentinel, choose and
   justify a handling strategy, encode at least one categorical column with
   an explicit judgment about its number of distinct values, standardize the
   numerical columns, and convert the result to a tensor. Report the shape
   and data type of the final tensor.

    *Adapted from UC Berkeley Data 100,
    [homework 2A "Food Safety"](https://github.com/DS-100/sp25-student/blob/main/hw/hw02A/hw02A.ipynb).*

1. [code] **Memory per row.** `data.info(memory_usage='deep')` reports
   the memory that the table occupies, text included. Compute the bytes per
   row of `data` as loaded, of `inputs` after imputation and one-hot
   encoding, and of the final `float32` tensor `X`. Which column costs most
   per row as loaded, and why? Then estimate how many homes a table of this
   kind could hold in 16 GB of main memory at each of the three stages.

[Discussions](https://d2l.discourse.group/t/195)